### The Winter Silence: Investigating the 2025/26 Transatlantic Mail Gap

At the end of 2025, something changed in the transatlantic mail stream. Between early November 2025 and mid-January 2026, the steady flow of postcards from Germany and Austria to the United States seemingly ground to a halt. While official postal data often remains opaque, the Postcrossing community — where German and US users form the backbone of global exchange — noticed the shift immediately.

Reports of "lost" mail began to pile up, with **travel times ballooning to over 60 days** without reaching the destination. For many, it felt as though months of holiday greetings had vanished into a "black hole." This notebook analyzes travel time distributions and volume shifts to uncover the extent of this anomaly.


In [3]:
import pandas as pd
import numpy as np
import plotly.express as px

In [6]:
df = pd.read_csv(
    "../data/processed/cleaned_data.csv", 
    parse_dates=['sent_date', 'received_date', 'extraction_date'],
    keep_default_na=False,  # Deactivate default NA handling --> NAMIBIA id (NA) as string
    na_values=[""]
)
print("Random sample of whole dataset:")
df.sample(10)


Random sample of whole dataset:


,country_id_dest,sent_date,received_date,distance_km,travel_time_days,extraction_date,country_id_origin,days_per_1000km,travel_route
1171612,BY,2015-10-12,2015-10-19,1431,7,2026-01-21,DE,4.89,DE-BY
735819,US,2013-07-29,2013-09-21,10185,54,2026-01-22,RU,5.30,RU-US
1416433,FI,2016-12-12,2017-02-20,2400,70,2026-01-22,FR,29.17,FR-FI
1168283,UA,2014-12-20,2015-01-07,1895,18,2026-01-21,DE,9.50,DE-UA
190272,CN,2013-07-20,2013-08-07,921,18,2026-01-17,TW,19.54,TW-CN
291445,DE,2018-01-17,2018-01-24,176,7,2026-01-21,NL,39.77,NL-DE
427604,FI,2014-01-24,2014-01-29,2074,5,2026-01-17,CH,2.41,CH-FI
1885515,US,2023-11-28,2023-12-05,1327,7,2026-01-17,US,5.28,US-US
68997,BY,2016-06-16,2016-06-25,813,9,2026-01-18,RU,11.07,RU-BY
289766,DE,2019-02-11,2019-03-13,1155,30,2026-01-21,LT,25.97,LT-DE


#### Where exactly is the US black hole?

While the "US black hole" became a heated topic primarily within the German-American community—driven by the sheer volume of active users on this route—a crucial question remains: Is this truly an isolated German-American issue, or are we looking at a much broader systemic failure?

To determine whether the silence is localized or part of a wider trend, we expanded our scope. By examining mail traffic from a selection of the world's most active regions, we can see if other global corridors are disappearing into the same void.


- USA (Domestic/Inbound)
- Canada (North America)
- Germany (Europe)
- China (Asia)
- Russia (Central Asia)
- Australia (Oceania)
- Brazil (South America)




In [7]:
#Filtering
country_selection_continents = ["US", "DE", "CA", "CN", "BR", "AU", "AU"]
df_us = df[df["country_id_dest"] == "US"]
df_us = df_us[df_us["country_id_origin"].isin(country_selection_continents)]
df_us = df_us[df_us["received_date"] <= df_us["extraction_date"].min()] # only postcards received before earliest extraction date 
df_us = df_us[df_us["sent_date"] >= pd.Timestamp("2015-01-01")] # only postcards sent since 2015
print(f"This dataset contains {len(df_us)} unique entries sent to the US.")
print(f"The here shown postcards were received latest on {df_us["received_date"].max().date()}.")



This dataset contains 99419 unique entries sent to the US.
The here shown postcards were received latest on 2026-01-17.


In [191]:
# Travel time

color_map = {
    "US": "#EF553B",  
    "DE": "#3943C9",  
    "CA": "#00CC96",  
    "CN": "#AB63FA",  
    "BR": "#FFA15A",  
    "AU": "#19D3F3",  
    "RU": "#D6CD45"    
}

monthly_data = df_us.groupby(['country_id_origin', pd.Grouper(key="sent_date", freq="MS")])['travel_time_days'].median().reset_index()
monthly_data['rolling_mean'] = monthly_data.groupby('country_id_origin')['travel_time_days'].transform(lambda x: x.rolling(window=6, center=True).mean())

fig = px.line(
    monthly_data,
    x="sent_date",
    y="rolling_mean",
    color="country_id_origin",
    title="How long do postcards to the US take to arrive? (6m-rolling mean of median)",
    labels={"sent_date": "Sent Date",
            "rolling_mean": "Travel Time (Median Days)",
            "country_id_origin": "Origin Country"},
    color_discrete_map=color_map,
    template="plotly_white",
    render_mode="svg"
)

fig.update_traces(mode="lines", marker=dict(size=4))
fig.update_layout(hovermode="x unified") 

fig.show();

##### From Seasonal Rhythms to Systemic Fractures

Historically, the transatlantic mail stream followed a predictable pulse. For a decade, the data showed only mild seasonal swells around the holidays, followed by a steady, gradual increase in global travel times. Even the massive disruptions of 2020 and 2021—the "COVID years"—felt like an anomaly that would eventually pass. However, as the world moved on, the old seasonal rhythms did not return; instead, they were eclipsed by deeper, more structural shifts in how mail moves across borders.

The first cracks in the system appeared between December 2023 and January 2024, when travel times spiked unexpectedly. This was the first echo of "Delivering for America," a sweeping USPS restructuring plan that left major International Service Centers (ISCs) struggling with significant backlogs. But the true turning point came in September 2024. With the implementation of the STOP Act and mandatory Electronic Advance Data (EAD) requirements, the flow of international mail hit a wall of new regulations. While most global routes eventually adapted and recovered, two corridors remained stubbornly clogged: Germany-to-US and Brazil-to-US.

The situation took a turn for the worse in May 2025. After a deceptive moment of stability during the summer, delays began to spiral again in September 2025. This time, the culprit was a "perfect storm" of stricter customs enforcement: the abolition of de-minimis rules and an aggressive crackdown on drug smuggling. These measures have pushed delivery times to record highs, leaving us with a lingering mystery: Why are Germany and Brazil bearing the brunt of these delays? To find out if this is a localized failure or a continental trend, we will now narrow our focus to the vast data landscape of the Europe-to-US route.

##### The European Fault Line: A Regional Glitch or a German Anomaly?
To understand if the "black hole" is swallowing all of Europe or merely targeting specific corridors, we must look beyond the German border. By comparing the mail volume received by US users from the most active European nations, we can pinpoint exactly where the flow of communication begins to fail.

Is the entire continent struggling under the weight of new US regulations, or is the breakdown suspiciously localized? We expanded our investigation to include the primary pulse points of European mail:

- Central Europe: Germany, Austria, Czech Republic, Poland
- Western Europe: The Netherlands, France, U.K.
- Northern Europe: Finland
- Eastern Europe & Eurasia: Belarus, Russian Federation

In [8]:
#Filtering
country_selection_europe = ["DE", "NL", "FI", "BY", "CZ", "PL", "GB", "FR", "AT", "RU"]
df_us = df[df["country_id_dest"] == "US"]
df_us = df_us[df_us["country_id_origin"].isin(country_selection_europe)]
df_us = df_us[df_us["received_date"] <= df_us["extraction_date"].min()] # only postcards received before earliest extraction date 
df_us = df_us[df_us["sent_date"] >= pd.Timestamp("2022-06-01")] # only postcards sent since 2022-06
print(f"This dataset contains {len(df_us)} unique entries sent to the US.")
print(f"The here shown postcards were received latest on {df_us["received_date"].max().date()}.")

This dataset contains 46991 unique entries sent to the US.
The here shown postcards were received latest on 2026-01-17.
